In [ ]:

import xarray as xr
import plotly
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from geopy.distance import geodesic
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import re
import glob
import imageio.v3 as iio
import subprocess
from joblib import delayed, Parallel
import numpy as np
import plotly.graph_objects as go
import plotly.graph_objects as go
from pace_pic_validation.viz.visualization import plot_pic_world


from pathlib import Path

In [ ]:
path_folder = '/Users/timfugl/Desktop/modist l3m PIC'
path_folder_daily = '/Users/timfugl/data/hermes_pic/ftp.hermes.acri.fr/560203259'
path_modis_0808_1208 = '/Volumes/Extreme SSD/PhD backup/Satellite Data/modis l3m PIC_08082026_120812026'
norbjornstations = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Research_Campaigns/Norbjørn /August 2026/data/Norbjorn_Stations.csv'



ds_nb = pd.read_csv(norbjornstations, sep=',', header=0, index_col=False)

ds_nb["Lat"] = ds_nb["Lat"].astype(float)
ds_nb["Long"] = ds_nb["Long"].astype(float)

print(ds_nb.head())

filenames = os.listdir(path_folder_daily)
filepaths = []

def merge_simple(folder, variable_name, start_index=None, end_index=None):

    filenames = os.listdir(folder)
    filepaths = []
    print(filepaths)
    for filename in filenames: 
        if filename.startswith('.'):
            continue  # skip hidden files
        path = os.path.join(folder, filename)
        print(path)
        filepaths.append(path)

    filepaths = filepaths[start_index:end_index]  # limit to first 10 files for testing

    print(len(filepaths))


# 2) Open all files and stack them along a new dimension (e.g. "file")
    ds = xr.open_mfdataset(
        filepaths,
        combine="nested",      # important when there's no natural coord like time
        concat_dim="file",     # new dimension name
        data_vars="all",       # or ["your_var"] to be explicit
        coords="all",
        compat="override",     # skip strict checks since lat/lon are identical
    )

# 3) Take the mean over the file dimension for your variable
    var_mean = ds[variable_name].mean(dim="file")


# 4) making a xarray dataset from var_mean
    var_mean_ds = var_mean.to_dataset(name=variable_name)

    print(var_mean_ds)

    return var_mean_ds


june_month = merge_simple(path_folder, 'pic', start_index = 0)

june_month.to_netcdf('/Users/timfugl/Desktop/modist l3m PIC/merged_june_month.nc')

print(june_month)

#lastdays = merge_simple(path_folder_daily, 'PIC_mean', start_index = 1, end_index =5 )


In [ ]:
import plotly.graph_objects as go

fig = plot_pic_world(
    june_month,
    var="pic",
    step=1,
    lat_min=52,
    lat_max=80,
    lon_min=-26,
    lon_max=20,
    title="MODIS PIC 26.7 to 8.8"
)


fig.show()

fig = plot_pic_world(
    lastdays,
    var="PIC_mean",
    step=1,
    lat_min = 52,
    lat_max = 80,
    lon_min = -26,
    lon_max = 20,
    title=" Globcolor PIC 1.8 to 4.8"
)

fig.show()


In [ ]:
#assign vars
sat = var_mean_ds
lat = sat['lat'].values
lon = sat['lon'].values   
PIC_8D = sat['pic_mean'].values


fig, ax = plt.subplots(figsize=(7, 7),
                       subplot_kw={'projection': ccrs.Stereographic()})

ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=0, alpha=0.2)
ax.add_feature(cfeature.LAND, facecolor='#90FF15', zorder=1, alpha=0.2)
ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.5, zorder=2)
ax.set_extent([-26, 30, 60, 85], crs=ccrs.PlateCarree())

    # Create 2D coordinate grids
lon_2d, lat_2d = np.meshgrid(lon, lat)

    # pcolormesh reprojects each cell correctly
im = ax.pcolormesh(lon_2d, lat_2d, PIC_8D,
                   transform=ccrs.PlateCarree(),
                   zorder=3, vmax=0.002, cmap='viridis')


plt.colorbar(im, ax=ax, label='PIC')
ax.set_title(f'PIC 26.7.2026 to 1.8.2026 MODIS')


plt.savefig('/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Research_Campaigns/Norbjørn /August 2026/Cruise Prep/PIC_MODIS_202607', dpi=300, bbox_inches='tight')
plt.show()

